# 🧾 chat_template の実描画チェック（install 不要）

Dataset の tokenizer でテンプレートを描画し、思考タグ・特殊トークンを確認。


In [ ]:
# chat_template を transformers で実際に描画する（vLLM 不要・1 分）
# 目的: GPU に上げる前に「テンプレートが壊れていない / 思考タグが正しい」を確認。
import json
import os

import transformers

ROOT = ("/kaggle/input/datasets/yamahei21python/"
        "qwen38-27b-uncensored-hauhaucs-awq-mtp")
print("transformers:", transformers.__version__)
print("files:", sorted(os.listdir(ROOT))[:20])

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(ROOT, trust_remote_code=True)
print("tokenizer:", type(tok).__name__, "vocab:", tok.vocab_size)
print("chat_template 長:", len(tok.chat_template or ""))
print("chat_template 先頭:", (tok.chat_template or "")[:300].replace("\n", " | "))

MSGS = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "1+1 は？"},
    {"role": "assistant", "content": "2 です。"},
    {"role": "user", "content": "では 2+2 は？"},
]

def ids_of(**kw):
    """tokenize=True は BatchEncoding を返す（len() は 2 になる）。input_ids を見る。"""
    out = tok.apply_chat_template(MSGS, tokenize=True, **kw)
    if isinstance(out, dict):
        return out["input_ids"]
    if hasattr(out, "input_ids"):
        return out.input_ids
    return out


for kw in ({}, {"add_generation_prompt": True}):
    try:
        text = tok.apply_chat_template(MSGS, tokenize=False, **kw)
        ids = ids_of(**kw)
        print(f"\n--- kw={kw} ---")
        print("chars:", len(text), "tokens:", len(ids))
        print(repr(text[-400:]))
        for tag in ("<|im_start|>", "<|im_end|>", "think", "tool_call"):
            print(f"  contains {tag!r}:", tag in text)
    except Exception as e:
        print(f"apply_chat_template NG kw={kw}: {type(e).__name__}: {e}")

# 特殊トークン ID の確認（reasoning parser / stop 条件に関係）
for name in ("eos_token", "pad_token", "bos_token"):
    t = getattr(tok, name, None)
    print(name, "=", t, getattr(t, "replace", lambda *a: None) and (tok.convert_tokens_to_ids(t) if t else None))
print("added_tokens:", len(tok.get_added_vocab()))

# ツール宣言がテンプレートで通るか（opencode 連携で効く）
TOOLS = [{"type": "function", "function": {
    "name": "get_weather", "description": "天気を返す",
    "parameters": {"type": "object",
                   "properties": {"city": {"type": "string"}},
                   "required": ["city"]}}}]
try:
    t2 = tok.apply_chat_template(MSGS, tools=TOOLS, tokenize=False,
                                 add_generation_prompt=True)
    print("\n--- tools ---")
    print("chars:", len(t2), "contains tool_call:", "tool_call" in t2)
    print(repr(t2[:600]))
except Exception as e:
    print("tools NG:", type(e).__name__, e)

# reasoning_effort がテンプレート変数か（system に xhigh が注入されていた）
tpl = tok.chat_template or ""
for kw in ("reasoning_effort", "xhigh", "effort"):
    print(f"template contains {kw!r}:", kw in tpl)
try:
    t3 = tok.apply_chat_template(MSGS, tokenize=False,
                                 add_generation_prompt=True,
                                 reasoning_effort="low")
    print("reasoning_effort=low chars:", len(t3))
    print(repr(t3[:300]))
except Exception as e:
    print("reasoning_effort NG:", type(e).__name__, e)

# generation_config の推奨サンプリング（RP / temp）
gen = json.load(open(os.path.join(ROOT, "generation_config.json")))
print("generation_config:", gen)
